# Template for the data structure 

We want to all use the same data structure as below


In [30]:
import numpy as np
import pandas as pd
import h5py

In [31]:
# Extend the HDF5 saving function to include metadata
def save_triplet_columns_and_metadata(df, triplet_columns, h5_filename):
    with h5py.File(h5_filename, "w") as f:
        # Save each triplet column as a dataset
        for col in triplet_columns:
            try:
                arr = np.vstack(df[col].values)  # Shape (N, 3)
                f.create_dataset(col, data=arr)
            except Exception as e:
                print(f"Skipping column '{col}' due to: {e}")
        
        # Save all other columns as metadata in JSON format
        metadata_cols = [col for col in df.columns if col not in triplet_columns]
        metadata_df = df[metadata_cols]

        # Convert to JSON-serializable format and store as a string
        meta_json = metadata_df.to_json(orient="records")
        f.create_dataset("metadata_json", data=np.bytes_(meta_json))
#


In [32]:
import pandas as pd

# Define columns, with key quantities stored as lists: [err-, value, err+]
columns = [
    "System Name", "RA", "Dec", "pos_err_mas", "Period", "Eccentricity",
    "M1","M1_sin3i", "M2", "M2_sin3i", "q", "Mass Function",
    "Type1", "Type2", "Detection Method", "Reference", "Notes"
]


# Initialize empty DataFrame
contact_df = pd.DataFrame(columns=columns)

# Define helper function using [err-, value, err+] triplets
def add_observation(df, system_name,
                    ra, dec, pos_err_mas, period, ecc,
                    m1, m1_sin3i, m2, m2_sin3i, q, mass_func,
                    type1, type2, method, reference, notes=""):
    # use np.inf if it is a lower limit (0 if it is an upper limit)
    new_row = {
       "System Name": system_name,
        "RA": ra,                           # scalar, deg
        "Dec": dec,                         # scalar, deg
        "pos_err_mas": pos_err_mas,         # 1-sigma on-sky positional uncertainty, mas

        "Period": period,                   # day [err-, value, err+]
        "Eccentricity": ecc,                # [err-, value, err+ ]

        "M1": m1,                           # Accretor star [err-, value, err+]
        "M1_sin3i": m1_sin3i,               # M1 sini^3 values [err-, value, err+] (= lower limit on m1)
        "M2": m2,                           # Donor (post MT 1) [err-, value, err+,]
        "M2_sin3i": m2_sin3i,               #  M2 sini^3 values [err-, value, err+] (= lower limit on m2)
        "q": q,                             # M2/M1 = donor/accretor [err-, value, err+, lower/upper limit? ]
        "Mass Function": mass_func,         # [err-, value, err+, lower/upper limit? ]

        "Type1": type1,                     # ["MS", "WD", "NS", "BH" "RG", "O", "B" ]
        "Type2": type2,                     # ["MS", "WD", "NS", "BH" "RG", "O", "B" ]
        "Detection Method": method,         # list of strings ["Xray", "RV"= Radial velocity, "EB"=Eclipsing binary, "AB" = Astrometric binary, "Other"]  
        "Reference": reference,             # ADS Bibcode    
        "Notes": notes
    }
    return pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

# Example: Add two entries
#observations_df = add_observation(
#    observations_df,
#    "Example system",
#    [0.005, 150.025, 0.005],        # RA
#    [0.004, -45.123, 0.004],        # Dec
#    [0.2, 12.5, 0.3],               # Period in days
#    [0.02, 0.30, 0.03],             # Eccentricity
##    [0.3, 10.0, 0.4],               # M1 in Msun
#    [0.2, 2.5, 0.3],                # M2 in Msun
#    [0.03, 0.25, 0.04],             # q (mass ratio)
#    [0.01, 1e-3, 0.02],             # Mass Function
#    "MS",                            # Type1
#    "WD",                            # Type2
#    ["RV", "EB"],                    # Detection Method
##    "2025arXiv250514780L",           # Reference (use Ads Bibcode)
#   "This is an example system , ",    # Notes
#)


##################
# Example edge cases

# How to add an lower limit (e.g., M1 > 5 Msun)?
#[0.0,5.0, np.inf],              
# Upper lower limit (e.g., M1 < 5 Msun)?
#[5.0, 5.0, 0.0],               
# # No error (e.g., M1 = 5 Msun)?
#[np.nan, 5.0, np.nan],            



In [33]:
contact_df = add_observation(contact_df,
    "LSS 3074", 			
    201.74923546386,     # RA (deg)
    -62.03037749215,     # Dec (deg)
    0.0118,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.0006, 2.1852, 0.0006],               # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [1.4, 17.2, 1.4],               # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [1.1, 14.8, 1.1],                # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.04, 0.86, 0.04],             # q (mass ratio)
    [np.nan,np.nan,np.nan],             # Mass Function
    "O6-7:(f):, Contact",           # Type1
    "O4 f +, Contact",                            # Type2
    ["RV", "EB"],                    # Detection Method
    ["2017A&A...601A.133R"],           # Reference (use Ads Bibcode)
    "Gaia DR3 5868409430865830912 , eccentricity set to 0 in modelling, eccentricity set to 0 in modelling",    # Notes
)


/var/folders/5d/vcxrsh5975l7d5n8t7vvc5pr0000gn/T/ipykernel_98496/2032401738.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)


In [34]:
contact_df = add_observation(contact_df,
    "MY Cam", 
    059.82622297558,     # RA (deg)
    +57.23715731461,     # Dec (deg)
    0.0236,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.0000015, 1.1754514, 0.0000015],               # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [1.6, 37.7, 1.6],               # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [1.4, 31.6, 1.4],                # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.04, 0.84, 0.04],             # q (mass ratio)
    [np.nan,np.nan,np.nan],             # Mass Function
    "O4.5-6V, Contact",           # Type1
    "O6V, Contact",                            # Type2
    ["RV", "EB"],                    # Detection Method
    ["2014A&A...572A.110L"],           # Reference (use Ads Bibcode)
    "Gaia DR3 469715181320008960 , eccentricity set to 0 in modelling",    # Notes
)

In [35]:
contact_df = add_observation(contact_df,
    "OGLE SMC-SC10 108086", 	
    016.37750625155,     # RA (deg)
    -72.02269945610,     # Dec (deg)
    0.0242,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [np.nan, 0.8830987, np.nan],               # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [1.2, 16.9, 1.2],               # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [1.7, 14.3, 1.7],               # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.06, 0.85, 0.06],             # q (mass ratio)
    [np.nan,np.nan,np.nan],        # Mass Function
    "O9V, Contact",                 # Type1
    "O9.5-B0V, Contact",                            # Type2
    ["RV", "EB"],                    # Detection Method
    ["2005MNRAS.357..304H"],           # Reference (use Ads Bibcode)
    "Gaia DR3 4690510870334496896, SMC , eccentricity set to 0 in modelling",    # Notes
)

In [36]:
contact_df = add_observation(contact_df,
    "TU Mus", 
    172.79544384537,     # RA (deg)
    -65.74225407400,     # Dec (deg)
    0.0195,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.0000017, 1.3872827, 0.0000017],               # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [0.4, 16.7, 0.4],               # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [0.4, 10.4, 0.4],               # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.009, 0.623, 0.009],             # q (mass ratio)
    [np.nan,np.nan,np.nan],        # Mass Function
    "O7V, Contact",                 # Type1
    "O8V, Contact",                            # Type2
    ["RV", "EB", "IUE"],                    # Detection Method
    ["2008ApJ...681..554P"],           # Reference (use Ads Bibcode)
    "Gaia DR3 5237207155697930496, HD 100213, eccentricity set to 0 in modelling",    # Notes
)

In [37]:
contact_df = add_observation(contact_df,
    "V382 Cyg", 	
    304.69673293063,     # RA (deg)
    +36.34054524438,     # Dec (deg)
    0.024,               # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.000007, 1.885545, 0.000007],               # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [0.4, 26.1, 0.4],               # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [0.3, 19.0, 0.3],               # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.005, 0.727,0.005],             # q (mass ratio)
    [np.nan,np.nan,np.nan],        # Mass Function
    "O6.5V((f)), Contact",                 # Type1
    "O6V((f)), Contact",                            # Type2
    ["RV", "EB"],                    # Detection Method
    ["2017A&A...607A..82M"],           # Reference (use Ads Bibcode)
    "Gaia DR3 2057530097777247744, HD 228854 , eccentricity set to 0 in modelling",    # Notes
)

In [38]:
contact_df = add_observation(contact_df,
    "VFTS 352", 		
    084.61858826377,     # RA (deg)
    -69.18864491570,     # Dec (deg)
    0.0195,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.0000004, 1.1241452, 0.0000004],               # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [0.3, 28.9, 0.3],               # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [0.3, 28.6, 0.3],               # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.10, 0.99, 0.10],             # q (mass ratio)
    [np.nan,np.nan,np.nan],        # Mass Function
    "O4.5 V(n)((fc))z, Contact",                 # Type1
    "O5.5 V(n)((fc))z, Contact",                            # Type2
    ["RV", "EB"],                    # Detection Method
    ["2015ApJ...812..102A"],           # Reference (use Ads Bibcode)
    "Gaia DR3 4657678216202797440, LMC, eccentricity set to 0 in modelling",    # Notes
)

In [39]:
contact_df = add_observation(contact_df,
    "CT Tau", 			
    089.70880588359,     # RA (deg)
    +27.07830392924,     # Dec (deg)
    0.021,               # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.00000016, 0.66682928, 0.00000016],               # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [3.3, 14.25, 3.3],               # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [3.4, 14.01, 3.4],               # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.003, 0.983, 0.003],             # q (mass ratio)
    [np.nan,np.nan,np.nan],        # Mass Function
    "B1V, Contact",                 # Type1
    "B1V, Contact",                            # Type2
    ["RV", "EB",],                    # Detection Method
    ["2019AJ....157..111Y"],           # Reference (use Ads Bibcode)
    "Gaia DR3 3430795519289951744,HD 249751, eccentricity set to 0 in modelling",    # Notes
)

In [40]:
contact_df = add_observation(contact_df,
    "GU Mon", 			
    101.19525090096,     # RA (deg)
    +00.22175161988,     # Dec (deg)
    0.0172,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.00000056, 0.89664680, 0.00000056],               # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [0.13, 8.79, 0.13],               # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [0.12, 8.58, 0.12],               # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.003, 0.976, 0.003],             # q (mass ratio)
    [np.nan,np.nan,np.nan],        # Mass Function
    "B1V, Contact",                 # Type1
    "B1V, Contact",                            # Type2
    ["RV", "EB",],                    # Detection Method
    ["2019AJ....157..111Y"],           # Reference (use Ads Bibcode)
    "Gaia DR3 3125506151511957120, eccentricity set to 0 in modelling",    # Notes
)

In [41]:
contact_df = add_observation(contact_df,
    "SV Cen", 		
    176.98835928994,     # RA (deg)
    -60.56604234441,     # Dec (deg)
    0.0235,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.001, 1.658, 0.001],               # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [0.35, 8.56, 0.35],               # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [0.26, 6.05, 0.26],               # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.002, 0.71, 0.002],             # q (mass ratio)
    [np.nan,np.nan,np.nan],        # Mass Function
    "B, Contact?",                 # Type1
    "B, Contact?",                            # Type2
    ["RV", "EB",],                    # Detection Method
    ["1992AJ....103..573R"],           # Reference (use Ads Bibcode)
    "Gaia DR3 5335388664983921024, HD 102552,Pdot significant, thermal timescale evolution?; might not be contact but semi-detached: 2024A&A...691A.150V', eccentricity set to 0 in modelling",    # Notes
)

In [42]:
contact_df = add_observation(contact_df,
    "V606 Cen", 
    200.40113235649,     # RA (deg)
    -60.52077138747,     # Dec (deg)
    0.011,               # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.0000011, 1.4950996, 0.0000011],               # Period in days
    [0.07, 0.0, 0.07],                # Eccentricity (tertiary has an eccentricity of 0.33, but the inner binary is likely circularized, so we set this to 0.0)
    [0.40, 14.70, 0.40],               # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [0.22, 7.74, 0.22],               # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.0007, 0.5484, 0.0007],             # q (mass ratio)
    [np.nan,np.nan,np.nan],        # Mass Function
    "B0.5V, Contact",                 # Type1
    "B2V, Contact",                            # Type2
    ["RV", "EB",],                    # Detection Method
    ["1999A&A...345..531L", "2022ApJ...924...30L"],           # Reference (use Ads Bibcode)
    "Gaia DR3 5869546428991037312, HD 115937; triple (P_out~88.3yr), eccentricity set to 0 in modelling",    # Notes
)

In [43]:
contact_df = add_observation(contact_df,
    "V701 Sco", 
    263.60214994049,     # RA (deg)
    -32.50444857968,     # Dec (deg)
    0.0293,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.0000013, 0.76187385, 0.0000013],               # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [0.22, 9.78, 0.22],               # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [0.22, 9.74, 0.22],               # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.002, 0.995, 0.002],             # q (mass ratio)
    [np.nan,np.nan,np.nan],        # Mass Function
    "B1-1.5V, Contact",                 # Type1
    "B1-1.5V, Contact",                            # Type2
    ["RV", "EB",],                    # Detection Method
    ["2019AJ....157..111Y"],           # Reference (use Ads Bibcode)
    "Gaia DR3 4054631788110501888, HD 317844, eccentricity set to 0 in modelling",    # Notes
)

In [44]:
contact_df = add_observation(contact_df,
    "V745 Cas", 	
    005.72221547173,     # RA (deg)
    +62.24137780568,     # Dec (deg)
    0.0919,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.000007, 1.410571, 0.000007],               # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [0.51, 18.31, 0.51],               # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [0.28, 10.47, 0.28],               # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.02, 0.57, 0.02],             # q (mass ratio)
    [np.nan,np.nan,np.nan],        # Mass Function
    "B0V, Contact",                 # Type1
    "B1-2 IV/V, Contact",                            # Type2
    ["RV", "EB",],                    # Detection Method
    ["2014MNRAS.442.1560C"],           # Reference (use Ads Bibcode)
    "Gaia DR3 430491104738148992, HD 1810, eccentricity set to 0 in modelling",    # Notes
)

In [45]:
contact_df = add_observation(contact_df,
    "VFTS 066", 
    084.38788320337,     # RA (deg)
    -69.07630004792,     # Dec (deg)
    0.0332,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.000007, 1.141171, 0.000007],               # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [5.0, 13.0, 7.0],               # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [2.8, 6.6, 3.5],               # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.014, 0.523, 0.014],             # q (mass ratio)
    [np.nan,np.nan,np.nan],        # Mass Function
    "B0V, Contact",                 # Type1
    "B1.5III, Contact",                            # Type2
    ["RV", "EB",],                    # Detection Method
    ["2020A&A...634A.119M"],           # Reference (use Ads Bibcode)
    "Gaia DR3 4657692406776887680, LMC, eccentricity set to 0 in modelling",    # Notes
)

In [46]:
contact_df = add_observation(contact_df,
    "RZ Pyx",
    133.01829794553,     # RA (deg)
    -27.48371579951,     # Dec (deg)
    0.0209,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.000034, 0.656273, 0.000034],          # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [0.4, 5.3, 0.4],                # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [0.10, 2.61, 0.10],             # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.02, 0.82, 0.02],             # q (mass ratio)
    [np.nan,np.nan,np.nan],         # Mass Function
    "B5V, Contact",                 # Type1
    "B5/7V, Contact",               # Type2
    ["RV", "EB",],                  # Detection Method
    ["1987MNRAS.227..481B"],          # Reference (use Ads Bibcode)
    "Gaia DR3 5648950448965966080, HD 75920, eccentricity set to 0 in modelling",    # Notes
)

In [47]:
contact_df = add_observation(contact_df,
    "LY Aur",		
    082.42771093332,     # RA (deg)
    +35.37501390189,     # Dec (deg)
    0.1388,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [np.nan, 4.02, np.nan],          # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [np.nan, 25.5, np.nan],                # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [np.nan, 14.0, np.nan],             # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.007, 0.550, 0.007],             # q (mass ratio)
    [np.nan,np.nan,np.nan],         # Mass Function
    "O9II, Contact",                 # Type1
    "O9III, Contact",               # Type2
    ["RV", "EB",],                  # Detection Method
    ["2013A&A...559A..22M"],          # Reference (use Ads Bibcode)
    "Gaia DR3 183255985260080896; quadruple, eccentricity set to 0 in modelling",    # Notes
)

In [48]:
contact_df = add_observation(contact_df, 
    "XZ Cep",	
    338.10447177545,     # RA (deg)
    +67.15067801114,     # Dec (deg)
    0.0155,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [np.nan, 5.097, np.nan],          # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [1.3, 18.7, 1.3],                # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [0.5,  9.3, 0.5],             # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.02, 0.50, 0.02],             # q (mass ratio)
    [np.nan,np.nan,np.nan],         # Mass Function
    "O9.5V, Contact",                 # Type1
    "B1III, Contact",               # Type2
    ["RV", "EB",],                  # Detection Method
    ["2017A&A...607A..82M"],          # Reference (use Ads Bibcode)
    "Gaia DR3 2224860133836098560, eccentricity set to 0 in modelling",    # Notes
)

In [49]:
contact_df = add_observation(contact_df,
    "V348 Car",
    156.74574995552,     # RA (deg)
    -57.67575034056,     # Dec (deg)
    0.014,               # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [np.nan, 5.562107, np.nan],          # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [4, 32, 4],                # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [4,  29, 4],             # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.02, 0.90, 0.02],             # q (mass ratio)
    [np.nan,np.nan,np.nan],         # Mass Function
    "B0-B.05III , Contact",                 # Type1
    "B1III, Contact",               # Type2
    ["RV", "EB",],                  # Detection Method
    ["1985MNRAS.213...75H"],          # Reference (use Ads Bibcode)
    "Gaia DR3 5351710915065605120; HD 90707, eccentricity set to 0 in modelling",    # Notes
)

In [50]:
contact_df = add_observation(contact_df,
    "V729 Cyg",	
    308.09341566800,     # RA (deg)
    +41.30524354553,     # Dec (deg)
    0.0198,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.000028, 6.597887, 0.000028],          # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [2.9, 31.6, 2.9],             # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [0.3,  8.8, 0.3],             # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.012, 0.290, 0.012],        # q (mass ratio)
    [np.nan,np.nan,np.nan],       # Mass Function
    "O6.5-7 Ia, Contact",                 # Type1
    "Ofpe/WN9, Contact",               # Type2
    ["RV", "EB",],                  # Detection Method
    ["2014NewA...31...32Y"],          # Reference (use Ads Bibcode)
    "Gaia DR3 2067830941174418048, eccentricity set to 0 in modelling",    # Notes
)

In [51]:
contact_df = add_observation(contact_df,
    "BH Cen",	
    174.79238064620,     # RA (deg)
    -63.42085538671,     # Dec (deg)
    0.0117,              # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [np.nan, 0.79159210, np.nan],          # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [5.4,  9.4, 5.4],             # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [5.4,  7.9, 5.4],             # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.003, 0.844, 0.003],        # q (mass ratio)
    [np.nan,np.nan,np.nan],       # Mass Function
    "B, Contact",                 # Type1
    "B, Contact",               # Type2
    ["RV", "EB",],                  # Detection Method
    ["1984AJ.....89..872L"],          # Reference (use Ads Bibcode)
    "Gaia DR3 5333576497997161728; HD 308826, eccentricity set to 0 in modelling",    # Notes
)

In [52]:
contact_df = add_observation(contact_df,
    "HD 64315B",		
    118.08449889505,     # RA (deg)
    -26.42962363296,     # Dec (deg)
    0.077,               # pos_err_mas (1-sigma on-sky error, mas; from Gaia/SIMBAD error ellipse)
    [0.0000008 , 1.0189569, 0.0000008 ],          # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [2.3,  14.6, 2.3],             # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [2.3,  14.6, 2.3],             # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.006, 1.00, 0.006],        # q (mass ratio)
    [np.nan,np.nan,np.nan],       # Mass Function
    "O9V, Contact",                 # Type1
    "O9V, Contact",               # Type2
    ["RV", "EB",],                  # Detection Method
    ["2017A&A...606A..54L"],          # Reference (use Ads Bibcode)
    "Gaia DR3 5602025904044961536; quadruple, eccentricity set to 0 in modelling",    # Notes
)

In [53]:
contact_df = add_observation(contact_df,
    "DIRECT V4741 M31A",		
    11.3338200,          # RA (deg) - SIMBAD, [VRJ2006] M31V-J00452011+4145037 (previous entry was wrong)
    41.7510500,          # Dec (deg)
    300,                 # pos_err_mas (~0.3 arcsec survey astrometry, Vilardell+2006; no SIMBAD error ellipse)
    [np.nan , 1.604, np.nan ],          # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [np.nan,  18, np.nan],             # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [np.nan,  16.8, np.nan],             # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.015, 0.924, 0.015],        # q (mass ratio)
    [np.nan,np.nan,np.nan],       # Mass Function
    "O, Contact",                 # Type1
    "O, Contact",               # Type2
    ["RV", "EB",],                  # Detection Method
    ["2022ApJ...932...14L"],          # Reference (use Ads Bibcode)
    "[VRJ2006] M31V-J00452011+4145037, M31, eccentricity set to 0 in modelling",    # Notes
)

In [54]:
contact_df = add_observation(contact_df,
    "DIRECT V1555 M31A",		
    11.2717800,          # RA (deg) - SIMBAD, LGGS J004505.24+413846.5 = M31V-J00450522+4138462 (previous RA was wrong)
    41.6461900,          # Dec (deg)
    300,                 # pos_err_mas (~0.3 arcsec survey astrometry, Vilardell+2006; no SIMBAD error ellipse)
    [np.nan , 0.917, np.nan ],          # Period in days
    [np.nan, 0.0, np.nan],             # Eccentricity
    [np.nan,  23, np.nan],             # M1 in Msun
    [np.nan,np.nan,np.nan],          # M1 sin3i
    [np.nan,  22.4, np.nan],             # M2 in Msun
    [np.nan,np.nan,np.nan],             # M2sin3 
    [0.012, 0.974, 0.012],        # q (mass ratio)
    [np.nan,np.nan,np.nan],       # Mass Function
    "O, Contact",                 # Type1
    "O, Contact",               # Type2
    ["RV", "EB",],                  # Detection Method
    ["2022ApJ...932...14L"],          # Reference (use Ads Bibcode)
    "[VRJ2006] M31V-J00450522+4138462, M31, eccentricity set to 0 in modelling",    # Notes
)

In [55]:
display(contact_df)

,System Name,RA,Dec,pos_err_mas,Period,Eccentricity,M1,M1_sin3i,M2,M2_sin3i,q,Mass Function,Type1,Type2,Detection Method,Reference,Notes
0,LSS 3074,201.749235,-62.030377,0.0118,"[0.0006, 2.1852, 0.0006]","[nan, 0.0, nan]","[1.4, 17.2, 1.4]","[nan, nan, nan]","[1.1, 14.8, 1.1]","[nan, nan, nan]","[0.04, 0.86, 0.04]","[nan, nan, nan]","O6-7:(f):, Contact","O4 f +, Contact","[RV, EB]",[2017A&A...601A.133R],"Gaia DR3 5868409430865830912 , eccentricity se..."
1,MY Cam,59.826223,57.237157,0.0236,"[1.5e-06, 1.1754514, 1.5e-06]","[nan, 0.0, nan]","[1.6, 37.7, 1.6]","[nan, nan, nan]","[1.4, 31.6, 1.4]","[nan, nan, nan]","[0.04, 0.84, 0.04]","[nan, nan, nan]","O4.5-6V, Contact","O6V, Contact","[RV, EB]",[2014A&A...572A.110L],"Gaia DR3 469715181320008960 , eccentricity set..."
2,OGLE SMC-SC10 108086,16.377506,-72.022699,0.0242,"[nan, 0.8830987, nan]","[nan, 0.0, nan]","[1.2, 16.9, 1.2]","[nan, nan, nan]","[1.7, 14.3, 1.7]","[nan, nan, nan]","[0.06, 0.85, 0.06]","[nan, nan, nan]","O9V, Contact","O9.5-B0V, Contact","[RV, EB]",[2005MNRAS.357..304H],"Gaia DR3 4690510870334496896, SMC , eccentrici..."
3,TU Mus,172.795444,-65.742254,0.0195,"[1.7e-06, 1.3872827, 1.7e-06]","[nan, 0.0, nan]","[0.4, 16.7, 0.4]","[nan, nan, nan]","[0.4, 10.4, 0.4]","[nan, nan, nan]","[0.009, 0.623, 0.009]","[nan, nan, nan]","O7V, Contact","O8V, Contact","[RV, EB, IUE]",[2008ApJ...681..554P],"Gaia DR3 5237207155697930496, HD 100213, eccen..."
4,V382 Cyg,304.696733,36.340545,0.0240,"[7e-06, 1.885545, 7e-06]","[nan, 0.0, nan]","[0.4, 26.1, 0.4]","[nan, nan, nan]","[0.3, 19.0, 0.3]","[nan, nan, nan]","[0.005, 0.727, 0.005]","[nan, nan, nan]","O6.5V((f)), Contact","O6V((f)), Contact","[RV, EB]",[2017A&A...607A..82M],"Gaia DR3 2057530097777247744, HD 228854 , ecce..."
5,VFTS 352,84.618588,-69.188645,0.0195,"[4e-07, 1.1241452, 4e-07]","[nan, 0.0, nan]","[0.3, 28.9, 0.3]","[nan, nan, nan]","[0.3, 28.6, 0.3]","[nan, nan, nan]","[0.1, 0.99, 0.1]","[nan, nan, nan]","O4.5 V(n)((fc))z, Contact","O5.5 V(n)((fc))z, Contact","[RV, EB]",[2015ApJ...812..102A],"Gaia DR3 4657678216202797440, LMC, eccentricit..."
6,CT Tau,89.708806,27.078304,0.0210,"[1.6e-07, 0.66682928, 1.6e-07]","[nan, 0.0, nan]","[3.3, 14.25, 3.3]","[nan, nan, nan]","[3.4, 14.01, 3.4]","[nan, nan, nan]","[0.003, 0.983, 0.003]","[nan, nan, nan]","B1V, Contact","B1V, Contact","[RV, EB]",[2019AJ....157..111Y],"Gaia DR3 3430795519289951744,HD 249751, eccent..."
7,GU Mon,101.195251,0.221752,0.0172,"[5.6e-07, 0.8966468, 5.6e-07]","[nan, 0.0, nan]","[0.13, 8.79, 0.13]","[nan, nan, nan]","[0.12, 8.58, 0.12]","[nan, nan, nan]","[0.003, 0.976, 0.003]","[nan, nan, nan]","B1V, Contact","B1V, Contact","[RV, EB]",[2019AJ....157..111Y],"Gaia DR3 3125506151511957120, eccentricity set..."
8,SV Cen,176.988359,-60.566042,0.0235,"[0.001, 1.658, 0.001]","[nan, 0.0, nan]","[0.35, 8.56, 0.35]","[nan, nan, nan]","[0.26, 6.05, 0.26]","[nan, nan, nan]","[0.002, 0.71, 0.002]","[nan, nan, nan]","B, Contact?","B, Contact?","[RV, EB]",[1992AJ....103..573R],"Gaia DR3 5335388664983921024, HD 102552,Pdot s..."
9,V606 Cen,200.401132,-60.520771,0.0110,"[1.1e-06, 1.4950996, 1.1e-06]","[0.07, 0.0, 0.07]","[0.4, 14.7, 0.4]","[nan, nan, nan]","[0.22, 7.74, 0.22]","[nan, nan, nan]","[0.0007, 0.5484, 0.0007]","[nan, nan, nan]","B0.5V, Contact","B2V, Contact","[RV, EB]","[1999A&A...345..531L, 2022ApJ...924...30L]","Gaia DR3 5869546428991037312, HD 115937; tripl..."


In [56]:
ref_list = [ref for ref in contact_df['Reference']]

from itertools import chain

flat_refs = list(chain.from_iterable(
    ref if isinstance(ref, list) else [ref]
    for ref in ref_list))
unique_refs = np.unique(flat_refs)

print(unique_refs)
 
for ref in unique_refs[5:]:
    print(f'https://articles.adsabs.harvard.edu/abs/{ref}')

['1984AJ.....89..872L' '1985MNRAS.213...75H' '1987MNRAS.227..481B'
 '1992AJ....103..573R' '1999A&A...345..531L' '2005MNRAS.357..304H'
 '2008ApJ...681..554P' '2013A&A...559A..22M' '2014A&A...572A.110L'
 '2014MNRAS.442.1560C' '2014NewA...31...32Y' '2015ApJ...812..102A'
 '2017A&A...601A.133R' '2017A&A...606A..54L' '2017A&A...607A..82M'
 '2019AJ....157..111Y' '2020A&A...634A.119M' '2022ApJ...924...30L'
 '2022ApJ...932...14L']
https://articles.adsabs.harvard.edu/abs/2005MNRAS.357..304H
https://articles.adsabs.harvard.edu/abs/2008ApJ...681..554P
https://articles.adsabs.harvard.edu/abs/2013A&A...559A..22M
https://articles.adsabs.harvard.edu/abs/2014A&A...572A.110L
https://articles.adsabs.harvard.edu/abs/2014MNRAS.442.1560C
https://articles.adsabs.harvard.edu/abs/2014NewA...31...32Y
https://articles.adsabs.harvard.edu/abs/2015ApJ...812..102A
https://articles.adsabs.harvard.edu/abs/2017A&A...601A.133R
https://articles.adsabs.harvard.edu/abs/2017A&A...606A..54L
https://articles.adsabs.harvard.ed

In [57]:
# RA/Dec are scalars now (with pos_err_mas); they go into metadata_json
triplet_cols = ["Period", "Eccentricity", "M1", "M1_sin3i", "M2", "M2_sin3i", "q", "Mass Function"]
save_triplet_columns_and_metadata(contact_df, triplet_cols, "../../../data/result_tables/legacy_h5/contact1.h5")


In [58]:
#contact_df.to_pickle('../contact.pkl')